# Lesson 02 - Explorer le Microsoft Agent Framework

Le **Microsoft Agent Framework (MAF)** est un cadre unifié pour construire des agents IA. Il offre une architecture claire et composable avec quatre blocs de construction principaux :

- **Client** – se connecte à un point de terminaison de modèle IA et gère la communication
- **Agent** – enveloppe un client avec des instructions et des définitions d’outils
- **Tools** – étendent les capacités de l’agent avec des fonctions personnalisées que le modèle peut appeler
- **Session** – maintient l’historique des conversations pour les interactions multi-tours

Dans cette leçon, nous allons créer un **agent de réservation de voyage** qui vérifie la disponibilité des destinations en utilisant ces concepts.


## Installation


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Comprendre l'architecture du framework Agent

Le framework Microsoft Agent suit une architecture en couches :

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – Un `AzureAIProjectAgentProvider` se connecte à un déploiement Azure OpenAI. Il gère l'authentification, le formatage des requêtes et l'analyse des réponses.
2. **Agent** – Créé à partir du client via `provider.create_agent()`, l'agent combine l'accès au modèle avec des instructions (invite système) et des outils.
3. **Outils** – Fonctions Python décorées avec `@tool` que l'agent peut invoquer pour effectuer des actions ou récupérer des données.
4. **Session** – Un objet `AgentSession` (créé via `agent.create_session()`) qui stocke l'historique de la conversation, permettant un dialogue à plusieurs tours où l'agent se souvient du contexte précédent.

Construisons chaque couche pas à pas.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Ajouter des outils avec le décorateur @tool

Les outils permettent aux agents d'effectuer des actions au-delà de la génération de texte. Le décorateur `@tool` transforme une fonction Python classique en quelque chose que l'agent peut appeler.

Points clés :
- Utilisez `Annotated[type, "description"]` pour que le modèle comprenne chaque paramètre.
- La docstring devient la description de l'outil que le modèle voit.
- `approval_mode="never_require"` signifie que l'outil s'exécute automatiquement sans confirmation de l'utilisateur.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Création d'un Agent avec des Outils

Nous combinons maintenant le client, les instructions et les outils en un agent. Les `instructions` agissent comme le prompt système — elles définissent la personnalité et le comportement de l'agent.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Conversations à plusieurs tours avec sessions

Une `AgentSession` (créée via `agent.create_session()`) conserve toutes les messages d'une conversation. En passant la même session à chaque appel `agent.run()`, l'agent a accès à l'historique complet de la conversation et peut se référer aux messages antérieurs.

Nous passons `tools=[check_destination_availability]` afin que l'agent puisse appeler notre vérificateur de disponibilité à chaque tour.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Résumé

Dans cette leçon, vous avez exploré les quatre piliers du Microsoft Agent Framework :

| Concept | Ce que vous avez appris |
|---------|------------------------|
| **Client** | `AzureAIProjectAgentProvider` se connecte à Azure OpenAI avec une authentification basée sur les informations d'identification |
| **Agent** | `provider.create_agent()` regroupe une connexion au modèle avec des instructions et un nom |
| **Outils** | Le décorateur `@tool` expose des fonctions Python que l'agent peut appeler |
| **Session** | `agent.create_session()` conserve l'historique des conversations sur plusieurs échanges |

Ces blocs de construction se combinent pour créer des agents capables de tenir des conversations naturelles, d'appeler des fonctions externes, et de maintenir le contexte — la base pour des modèles agentiques plus avancés dans les leçons suivantes.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Clause de non-responsabilité** :  
Ce document a été traduit à l’aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforcions d’assurer l’exactitude, veuillez noter que les traductions automatiques peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue d’origine doit être considéré comme la source faisant foi. Pour les informations critiques, il est recommandé de faire appel à une traduction professionnelle réalisée par un humain. Nous ne saurions être tenus responsables de tout malentendu ou mauvaise interprétation résultant de l’utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
